# Tablero de completitud de cuentas contables — DWH

**¿Qué hace este notebook?** Se conecta al Data Warehouse (DWH) y revisa, para el **último año
(12 meses)**, si el campo `VALOR_RESERVA_CONTABLE` (la plata que la aseguradora aparta como
respaldo) tiene valor o está vacío, por cada combinación de **fuente + cuenta contable + libro**.

Es la versión con historia del control `control_completitud_cuentas.sql` (que revisa un solo mes).

**¿Cuándo correrlo?** Los días **8 o 9** de cada mes, antes del proceso oficial de cierre (día 10),
para poder avisar a tiempo si a alguna cuenta le falta información.

**Cómo leer el resultado (semáforo):**
- 🟢 **COMPLETO** = todos los registros de esa cuenta/libro/mes tienen valor → OK para cerrar.
- 🔴 **ALERTA** = hay registros con el valor vacío (nulo) → avisar al dueño de la tabla.

**Cuentas que NO se revisan aquí** (tienen sus propios controles): Incurrido, Salvamentos,
Recobros, IVA AG (se excluye con el filtro `Libro <> 'AG'`) + 2 cuentas por confirmar con Andrey.

> Las credenciales se leen de `credenciales_local.py`, que está en el `.gitignore`
> y **nunca** debe subirse al repositorio.

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

# Credenciales locales (archivo ignorado por git, vive solo en este computador)
from credenciales_local import (
    DEFAULT_HOST_DWH, DEFAULT_PORT_DWH, DEFAULT_DATABASE_DWH,
    DEFAULT_USER_DWH, DEFAULT_PASSWORD_DWH,
)

## 1. Conexión al DWH

Creamos la conexión al servidor SQL Server del Data Warehouse y hacemos una prueba mínima
(`SELECT 1`): si la celda imprime `conexion_ok = 1`, la conexión funciona.

In [12]:
url = URL.create(
    "mssql+pyodbc",
    username=DEFAULT_USER_DWH,
    password=DEFAULT_PASSWORD_DWH,
    host=DEFAULT_HOST_DWH,
    port=int(DEFAULT_PORT_DWH),
    database=DEFAULT_DATABASE_DWH,
    query={"driver": "SQL Server"},  # driver ODBC disponible en esta máquina
)
engine = create_engine(url)

pd.read_sql("SELECT 1 AS conexion_ok", engine)

,conexion_ok
0,1


## 2. ¿Qué meses vamos a revisar?

En vez de escribir el periodo a mano, le preguntamos a las propias tablas cuál es el **último
periodo contable cargado** (formato `AAAAMM`, por ejemplo `202607` = julio de 2026) y desde ahí
contamos **12 meses hacia atrás**. Así el tablero siempre muestra el último año disponible sin
tener que editar nada.

> Solo se consulta el máximo en `DIRECTA` y `CEDIDAS_IAXIS`, que responden en segundos.
> Las otras dos tablas (`CEDIDAS_AS400` y `CEDIDAS_TERREMOTO`) tardan varios minutos incluso
> para un simple `MAX` (no tienen índice por periodo), y comparten el mismo calendario de carga.

In [13]:
sql_ultimo_periodo = """
SELECT MAX(p) AS ultimo_periodo
FROM (
    SELECT MAX(periodo_contable_analisis) AS p FROM Liberty.RESERVAS.DIRECTA_RESERVA_INTERFAZ
    UNION ALL
    SELECT MAX(periodo_contable_analisis)      FROM Liberty.RESERVAS.CEDIDAS_RESERVA_INTERFAZ_IAXIS
) t
"""
ULTIMO_PERIODO = int(pd.read_sql(sql_ultimo_periodo, engine).iloc[0, 0])


def periodos_hacia_atras(periodo_final: int, n: int = 12) -> list[int]:
    """Lista de n periodos AAAAMM terminando en periodo_final (incluido)."""
    anio, mes = divmod(periodo_final, 100)
    periodos = []
    for _ in range(n):
        periodos.append(anio * 100 + mes)
        mes -= 1
        if mes == 0:
            anio, mes = anio - 1, 12
    return sorted(periodos)


PERIODOS = periodos_hacia_atras(ULTIMO_PERIODO, 12)
print(f"Último periodo cargado en DWH: {ULTIMO_PERIODO}")
print(f"Ventana de análisis (12 meses): {PERIODOS[0]} → {PERIODOS[-1]}")

Último periodo cargado en DWH: 202606
Ventana de análisis (12 meses): 202507 → 202606


## 3. El control de completitud (12 meses en una sola consulta)

Es exactamente la misma lógica de `control_completitud_cuentas.sql` — mismas 4 fuentes, mismas
cuentas, mismos filtros (`Libro <> 'AG'`, etc.) — con un solo cambio: en vez de revisar **un**
periodo, revisa los **12** de la ventana.

Columnas del resultado:
- `total_registros`: cuántas filas hay para esa cuenta/libro/mes.
- `con_valor` / `sin_valor`: cuántas tienen el valor de la reserva y cuántas lo tienen vacío (nulo).
- `pct_completitud`: porcentaje con valor (100 = perfecto).
- `semaforo`: COMPLETO o ALERTA.

> ⏳ **Paciencia**: esta celda puede tardar **varios minutos** — dos de las tablas no tienen
> índice por periodo y el motor tiene que recorrerlas completas. Es normal.

In [14]:
# Los periodos son enteros calculados por nosotros (no texto del usuario),
# por eso es seguro insertarlos directamente en el SQL.
lista_periodos = ", ".join(str(int(p)) for p in PERIODOS)

query_control = f"""
SELECT
    fuente,
    cuenta,
    libro,
    periodo_contable_analisis,
    total_registros,
    con_valor,
    sin_valor,
    CAST(ROUND(100.0 * con_valor / NULLIF(total_registros, 0), 2) AS DECIMAL(5,2)) AS pct_completitud,
    semaforo
FROM
(
    /* 1. DIRECTA — IAXIS */
    SELECT
        'DIRECTA_IAXIS' AS fuente,
        a.CUENTA        AS cuenta,
        a.Libro         AS libro,
        a.periodo_contable_analisis,
        COUNT(*)        AS total_registros,
        SUM(CASE WHEN a.VALOR_RESERVA_CONTABLE IS NOT NULL THEN 1 ELSE 0 END) AS con_valor,
        SUM(CASE WHEN a.VALOR_RESERVA_CONTABLE IS NULL     THEN 1 ELSE 0 END) AS sin_valor,
        CASE WHEN SUM(CASE WHEN a.VALOR_RESERVA_CONTABLE IS NULL THEN 1 ELSE 0 END) = 0
             THEN 'COMPLETO' ELSE 'ALERTA' END AS semaforo
    FROM Liberty.RESERVAS.DIRECTA_RESERVA_INTERFAZ a
    WHERE a.periodo_contable_analisis IN ({lista_periodos})
      AND a.CUENTA IN ('410305','410310','410315','510305','510310','510315')
      AND a.Libro <> 'AG'
    GROUP BY a.CUENTA, a.Libro, a.periodo_contable_analisis

    UNION ALL

    /* 2. CEDIDAS — AS400 */
    SELECT
        'CEDIDAS_AS400' AS fuente,
        a.CUENTA, a.LIBRO, a.periodo_contable_analisis,
        COUNT(*),
        SUM(CASE WHEN a.VALOR_RESERVA_CONTABLE IS NOT NULL THEN 1 ELSE 0 END),
        SUM(CASE WHEN a.VALOR_RESERVA_CONTABLE IS NULL     THEN 1 ELSE 0 END),
        CASE WHEN SUM(CASE WHEN a.VALOR_RESERVA_CONTABLE IS NULL THEN 1 ELSE 0 END) = 0
             THEN 'COMPLETO' ELSE 'ALERTA' END
    FROM Liberty.RESERVAS.CEDIDAS_RESERVA_INTERFAZ a
    WHERE a.periodo_contable_analisis IN ({lista_periodos})
      AND a.CUENTA IN ('510305','410305')
      AND a.LIBRO <> 'AG'
      AND a.RAMO_PROD IS NOT NULL
    GROUP BY a.CUENTA, a.LIBRO, a.periodo_contable_analisis

    UNION ALL

    /* 3. CEDIDAS — IAXIS */
    SELECT
        'CEDIDAS_IAXIS' AS fuente,
        a.CUENTA, a.LIBRO, a.periodo_contable_analisis,
        COUNT(*),
        SUM(CASE WHEN a.VALOR_RESERVA_CONTABLE IS NOT NULL THEN 1 ELSE 0 END),
        SUM(CASE WHEN a.VALOR_RESERVA_CONTABLE IS NULL     THEN 1 ELSE 0 END),
        CASE WHEN SUM(CASE WHEN a.VALOR_RESERVA_CONTABLE IS NULL THEN 1 ELSE 0 END) = 0
             THEN 'COMPLETO' ELSE 'ALERTA' END
    FROM Liberty.RESERVAS.CEDIDAS_RESERVA_INTERFAZ_IAXIS a
    WHERE a.periodo_contable_analisis IN ({lista_periodos})
      AND a.CUENTA IN ('510305','410305')
      AND a.LIBRO <> 'AG'
      AND a.RAMO_PROD IS NOT NULL
    GROUP BY a.CUENTA, a.LIBRO, a.periodo_contable_analisis

    UNION ALL

    /* 4. CEDIDAS TERREMOTO */
    SELECT
        'CEDIDAS_TERREMOTO' AS fuente,
        a.CUENTA,
        CAST(a.LIBRO AS VARCHAR(20)),
        a.periodo_contable_analisis,
        COUNT(*),
        SUM(CASE WHEN a.VALOR_RESERVA_CONTABLE IS NOT NULL THEN 1 ELSE 0 END),
        SUM(CASE WHEN a.VALOR_RESERVA_CONTABLE IS NULL     THEN 1 ELSE 0 END),
        CASE WHEN SUM(CASE WHEN a.VALOR_RESERVA_CONTABLE IS NULL THEN 1 ELSE 0 END) = 0
             THEN 'COMPLETO' ELSE 'ALERTA' END
    FROM Liberty.RESERVAS.CEDIDAS_TERREMOTO_RESERVA_INTERFAZ a
    WHERE a.periodo_contable_analisis IN ({lista_periodos})
      AND UPPER(LTRIM(RTRIM(CAST(a.FUENTE_INTERFAZ AS VARCHAR(50))))) = 'TERR'
      AND ISNULL(LTRIM(RTRIM(CAST(a.LIBRO AS VARCHAR(20)))), '') <> 'AG'
    GROUP BY a.CUENTA, a.LIBRO, a.periodo_contable_analisis
) resumen
"""

df = pd.read_sql(query_control, engine)
df["pct_completitud"] = df["pct_completitud"].astype(float)
df["pct_sin_valor"] = (100.0 * df["sin_valor"] / df["total_registros"]).round(2)

print(f"{len(df)} filas (combinaciones fuente + cuenta + libro + mes)")
df.head(10)

240 filas (combinaciones fuente + cuenta + libro + mes)


,fuente,cuenta,libro,periodo_contable_analisis,total_registros,con_valor,sin_valor,pct_completitud,semaforo,pct_sin_valor
0,DIRECTA_IAXIS,410310,AL,202603,1513565,1513565,0,100.0,COMPLETO,0.0
1,DIRECTA_IAXIS,410315,AL,202604,81474,81474,0,100.0,COMPLETO,0.0
2,DIRECTA_IAXIS,510305,AL,202603,8105886,8105886,0,100.0,COMPLETO,0.0
3,DIRECTA_IAXIS,510305,AL,202604,9259457,9259457,0,100.0,COMPLETO,0.0
4,DIRECTA_IAXIS,510310,AL,202604,2143781,2143781,0,100.0,COMPLETO,0.0
5,DIRECTA_IAXIS,510315,AL,202507,59243,59243,0,100.0,COMPLETO,0.0
6,DIRECTA_IAXIS,410305,AL,202508,3243169,3243169,0,100.0,COMPLETO,0.0
7,DIRECTA_IAXIS,410305,AL,202603,8976620,8976620,0,100.0,COMPLETO,0.0
8,DIRECTA_IAXIS,410305,AL,202605,7286000,7286000,0,100.0,COMPLETO,0.0
9,DIRECTA_IAXIS,510310,AL,202603,1919823,1919823,0,100.0,COMPLETO,0.0


## 4. Reglas de calidad adicionales — variación y volumen

Hasta acá el tablero solo revisaba **si el dato existe** (completitud). Ahora agregamos una revisión
distinta: **¿se movió más de lo normal?** — esto es justo lo que en `Hallazgos y Estado del Proyecto.md`
se llama la "Fase 2" (detectar valores atípicos), que todavía no estaba definida. Lo que sigue es una
primera propuesta simple; valídala con el equipo antes de confiar en ella del todo.

Vamos a revisar tres cosas nuevas:

- **4.1 Variación de valor**: ¿la plata total de la reserva contable de una cuenta/libro cambió mucho
  de un mes a otro?
- **4.2 Variación de volumen**: ¿la cantidad de registros (filas) cambió mucho? A veces el problema no
  es el valor sino que "desaparecieron" o "aparecieron" registros de la nada.
- **4.3 Huecos de carga**: ¿hubo algún mes en que una cuenta/libro no tuvo **ni una sola fila**? Esto
  casi siempre significa que esa carga no corrió ese mes, más que un problema del dato en sí.

Una combinación se marca como **"variación atípica"** si pasa cualquiera de estas dos pruebas:

1. Se aleja más de **2 desviaciones estándar** de su propio promedio en la ventana de 12 meses (el
   "z-score" — no te preocupes por la fórmula exacta: el número solo dice qué tan raro es este valor
   comparado con lo que esa misma cuenta suele tener).
2. Cambió más de **20%** respecto al mes inmediatamente anterior.

Ambos umbrales están definidos como variables (`UMBRAL_Z_VARIACION`, `UMBRAL_PCT_VARIACION`) al inicio
del siguiente bloque de código — súbelos si el tablero avisa demasiado, o bájalos si avisa muy poco.

In [15]:
df["serie"] = df["fuente"] + " · " + df["cuenta"].astype(str) + " · " + df["libro"].astype(str).str.strip()

# UMBRALES: primera propuesta, hay que validarlos con el equipo — esto es la "Fase 2"
# que menciona Hallazgos y Estado del Proyecto.md y que todavía no estaba definida.
UMBRAL_Z_VARIACION = 2.0     # desviaciones estándar respecto al promedio de la propia serie
UMBRAL_PCT_VARIACION = 0.20  # 20% de cambio respecto al mes inmediatamente anterior


def marcar_variacion_atipica(pivot: pd.DataFrame, periodos: list[int],
                              umbral_z: float = UMBRAL_Z_VARIACION,
                              umbral_pct: float = UMBRAL_PCT_VARIACION,
                              nombre_id: str = "serie") -> pd.DataFrame:
    """
    Recibe una tabla (serie x periodo) y devuelve, en formato largo, una fila por
    serie+periodo con: el valor, su z-score respecto al promedio de esa misma serie
    en la ventana, la variación % contra el mes anterior, y si se marca atípica.
    """
    promedio = pivot.mean(axis=1)
    desviacion = pivot.std(axis=1)
    filas = []
    for idx in pivot.index:
        valores = pivot.loc[idx]
        for i, periodo in enumerate(periodos):
            valor = valores.get(periodo)
            if pd.isna(valor):
                continue
            d = desviacion[idx]
            z = (valor - promedio[idx]) / d if d and not pd.isna(d) else float("nan")
            anterior = valores.get(periodos[i - 1]) if i > 0 else None
            if anterior is None or pd.isna(anterior):
                pct = float("nan")
            elif anterior == 0:
                pct = float("inf") if valor != 0 else 0.0
            else:
                pct = (valor - anterior) / abs(anterior)
            atipica = (not pd.isna(z) and abs(z) > umbral_z) or (not pd.isna(pct) and abs(pct) > umbral_pct)
            filas.append({
                nombre_id: idx,
                "periodo_contable_analisis": periodo,
                "valor": valor,
                "z_score": z,
                "pct_variacion": pct,
                "variacion_atipica": bool(atipica),
            })
    return pd.DataFrame(filas)

### 4.1 Variación de valor

Necesitamos una consulta nueva: la de la sección 3 solo cuenta cuántos registros tienen valor o no
(para completitud), pero no suma el valor en sí. Aquí sumamos `VALOR_RESERVA_CONTABLE` por fuente +
cuenta + libro + mes — misma estructura de 4 fuentes y mismos filtros que la sección 3, solo que
sumando en vez de contando nulos.

In [16]:
query_variacion = f"""
SELECT
    fuente,
    cuenta,
    libro,
    periodo_contable_analisis,
    valor_total
FROM
(
    /* 1. DIRECTA — IAXIS */
    SELECT
        'DIRECTA_IAXIS' AS fuente,
        a.CUENTA        AS cuenta,
        a.Libro         AS libro,
        a.periodo_contable_analisis,
        SUM(CAST(ISNULL(a.VALOR_RESERVA_CONTABLE, 0) AS DECIMAL(28,6))) AS valor_total
    FROM Liberty.RESERVAS.DIRECTA_RESERVA_INTERFAZ a
    WHERE a.periodo_contable_analisis IN ({lista_periodos})
      AND a.CUENTA IN ('410305','410310','410315','510305','510310','510315')
      AND a.Libro <> 'AG'
    GROUP BY a.CUENTA, a.Libro, a.periodo_contable_analisis

    UNION ALL

    /* 2. CEDIDAS — AS400 */
    SELECT
        'CEDIDAS_AS400' AS fuente,
        a.CUENTA, a.LIBRO, a.periodo_contable_analisis,
        SUM(CAST(ISNULL(a.VALOR_RESERVA_CONTABLE, 0) AS DECIMAL(28,6)))
    FROM Liberty.RESERVAS.CEDIDAS_RESERVA_INTERFAZ a
    WHERE a.periodo_contable_analisis IN ({lista_periodos})
      AND a.CUENTA IN ('510305','410305')
      AND a.LIBRO <> 'AG'
      AND a.RAMO_PROD IS NOT NULL
    GROUP BY a.CUENTA, a.LIBRO, a.periodo_contable_analisis

    UNION ALL

    /* 3. CEDIDAS — IAXIS */
    SELECT
        'CEDIDAS_IAXIS' AS fuente,
        a.CUENTA, a.LIBRO, a.periodo_contable_analisis,
        SUM(CAST(ISNULL(a.VALOR_RESERVA_CONTABLE, 0) AS DECIMAL(28,6)))
    FROM Liberty.RESERVAS.CEDIDAS_RESERVA_INTERFAZ_IAXIS a
    WHERE a.periodo_contable_analisis IN ({lista_periodos})
      AND a.CUENTA IN ('510305','410305')
      AND a.LIBRO <> 'AG'
      AND a.RAMO_PROD IS NOT NULL
    GROUP BY a.CUENTA, a.LIBRO, a.periodo_contable_analisis

    UNION ALL

    /* 4. CEDIDAS TERREMOTO */
    SELECT
        'CEDIDAS_TERREMOTO' AS fuente,
        a.CUENTA,
        CAST(a.LIBRO AS VARCHAR(20)),
        a.periodo_contable_analisis,
        SUM(CAST(ISNULL(a.VALOR_RESERVA_CONTABLE, 0) AS DECIMAL(28,6)))
    FROM Liberty.RESERVAS.CEDIDAS_TERREMOTO_RESERVA_INTERFAZ a
    WHERE a.periodo_contable_analisis IN ({lista_periodos})
      AND UPPER(LTRIM(RTRIM(CAST(a.FUENTE_INTERFAZ AS VARCHAR(50))))) = 'TERR'
      AND ISNULL(LTRIM(RTRIM(CAST(a.LIBRO AS VARCHAR(20)))), '') <> 'AG'
    GROUP BY a.CUENTA, a.LIBRO, a.periodo_contable_analisis
) resumen
"""

df_valor = pd.read_sql(query_variacion, engine)
df_valor["serie"] = (df_valor["fuente"] + " · " + df_valor["cuenta"].astype(str)
                     + " · " + df_valor["libro"].astype(str).str.strip())

piv_valor = (df_valor.pivot_table(index="serie", columns="periodo_contable_analisis",
                                  values="valor_total", aggfunc="sum")
             .reindex(columns=PERIODOS))

variacion_valor = marcar_variacion_atipica(piv_valor, PERIODOS)
alertas_valor = (variacion_valor[variacion_valor["variacion_atipica"]]
                 .sort_values("periodo_contable_analisis", ascending=False)
                 .reset_index(drop=True))

print(f"{len(df_valor)} filas de valor total (fuente + cuenta + libro + mes)")
print(f"{len(alertas_valor)} variaciones atípicas de VALOR en la ventana de 12 meses")
alertas_valor.head(10)

240 filas de valor total (fuente + cuenta + libro + mes)
153 variaciones atípicas de VALOR en la ventana de 12 meses


,serie,periodo_contable_analisis,valor,z_score,pct_variacion,variacion_atipica
0,CEDIDAS_AS400 · 510305 · AA,202606,8.083954e+08,1.409595,3.575248,True
1,CEDIDAS_IAXIS · 510305 · AL,202606,4.441608e+08,0.373344,-0.666169,True
2,DIRECTA_IAXIS · 510310 · AA,202606,2.259010e+09,1.365615,0.593122,True
3,CEDIDAS_TERREMOTO · 510305 · AL,202606,-5.958562e+08,0.191155,0.868762,True
4,CEDIDAS_TERREMOTO · 510305 · AA,202606,-4.896680e+08,-1.120584,-3.142477,True
5,CEDIDAS_TERREMOTO · 410305 · AL,202606,1.135412e+09,0.244918,-0.290825,True
6,CEDIDAS_TERREMOTO · 410305 · AA,202606,-5.096016e+06,0.037570,-1.002011,True
7,CEDIDAS_TERREMOTO · 263005 · AL,202606,-5.395555e+08,-0.507411,-1.183569,True
8,CEDIDAS_TERREMOTO · 263005 · AA,202606,4.947640e+08,0.286739,1.179068,True
9,DIRECTA_IAXIS · 410310 · AA,202606,-7.624850e+08,0.182717,0.231039,True


### 4.2 Variación de volumen

No necesitamos otra consulta: la columna `total_registros` ya viene en el `df` de la sección 3. Aquí
solo aplicamos la misma prueba de variación atípica, pero sobre la cantidad de filas en vez del valor
monetario.

In [17]:
piv_vol = (df.pivot_table(index="serie", columns="periodo_contable_analisis",
                          values="total_registros", aggfunc="sum")
           .reindex(columns=PERIODOS))

variacion_vol = marcar_variacion_atipica(piv_vol, PERIODOS)
alertas_volumen = (variacion_vol[variacion_vol["variacion_atipica"]]
                   .sort_values("periodo_contable_analisis", ascending=False)
                   .reset_index(drop=True))

print(f"{len(alertas_volumen)} variaciones atípicas de VOLUMEN (cantidad de registros) en la ventana de 12 meses")
alertas_volumen.head(10)

83 variaciones atípicas de VOLUMEN (cantidad de registros) en la ventana de 12 meses


,serie,periodo_contable_analisis,valor,z_score,pct_variacion,variacion_atipica
0,CEDIDAS_TERREMOTO · 263005 · AL,202606,90568,-0.484468,-0.258709,True
1,CEDIDAS_TERREMOTO · 263005 · AA,202606,32,-0.688636,-0.959698,True
2,CEDIDAS_TERREMOTO · 410305 · AA,202606,1,-0.651650,-0.998561,True
3,CEDIDAS_TERREMOTO · 510305 · AL,202606,26503,-0.190418,0.210293,True
4,CEDIDAS_TERREMOTO · 510305 · AA,202606,31,-0.556228,-0.686869,True
5,CEDIDAS_TERREMOTO · 410305 · AL,202606,64065,-0.351629,-0.361126,True
6,DIRECTA_IAXIS · 410310 · AA,202606,9570,-0.229576,-0.250587,True
7,CEDIDAS_TERREMOTO · 510305 · AL,202605,21898,-0.299016,0.871304,True
8,DIRECTA_IAXIS · 510315 · AL,202605,29679,-0.619880,0.213071,True
9,CEDIDAS_TERREMOTO · 510305 · AA,202604,90,-0.553861,1.432432,True


### 4.3 Huecos de carga

Un "hueco" es distinto de una "alerta de completitud": una alerta significa que llegaron filas pero
algunas tienen el valor vacío; un hueco significa que **no llegó ninguna fila** para esa cuenta/libro
en ese mes — probablemente la carga de esa fuente no corrió.

In [18]:
piv_tot = (df.pivot_table(index="serie", columns="periodo_contable_analisis",
                          values="total_registros", aggfunc="sum")
           .reindex(columns=PERIODOS))

huecos = (piv_tot.reset_index()
          .melt(id_vars="serie", var_name="periodo_contable_analisis", value_name="total_registros"))
huecos = (huecos[huecos["total_registros"].isna()]
          .drop(columns="total_registros")
          .sort_values(["serie", "periodo_contable_analisis"])
          .reset_index(drop=True))

print(f"{len(huecos)} huecos de carga (combinación fuente+cuenta+libro sin ninguna fila en ese mes)")
huecos.head(10)

0 huecos de carga (combinación fuente+cuenta+libro sin ninguna fila en ese mes)


,serie,periodo_contable_analisis


## 5. Tablero HTML estilo HDI — variación combinada + plantilla corporativa

El tablero anterior (con gráficos Plotly y un mapa de calor) no encajaba con el estilo que ya usa el
equipo en otros tableros de calidad de datos. Esta sección arma un tablero nuevo, reutilizando la
misma plantilla visual (`notebooks/tablero_template.html`) que ya se usó para el tablero de KPIs de
Calidad de Datos de CDP Primas: aros de KPI, pestañas, tablas semáforo, tooltips, y un modal de
detalle al hacer clic en un aro.

Para encajar en esa plantilla, los datos se organizan en un formato genérico de "indicadores": cada
fila es una combinación de `tipo_indicador` (Completitud / Variación / Disponibilidad) + `nombre_campo`
(nuestra combinación fuente · cuenta · libro) + `periodo_contable` + un porcentaje de 0 a 100.

También simplificamos: en vez de mostrar variación de **valor** y de **volumen** por separado (como en
la sección 4), aquí las combinamos en un solo indicador, **"Variación (Estabilidad)"**: una combinación
se considera inestable si **cualquiera** de las dos señales (valor o volumen) se sale de lo normal —
así no hay que revisar dos pestañas distintas para lo mismo.

In [ ]:
# K_PCT y K_Z convierten z-score / % de variación en un puntaje 0-100 ("estabilidad"),
# calibrados para que el puntaje caiga exactamente a 80 (el límite de ALERTA que
# definimos más abajo) cuando se cruza cualquiera de los dos umbrales de la sección 4.
K_PCT = 20.0 / UMBRAL_PCT_VARIACION
K_Z = 20.0 / UMBRAL_Z_VARIACION


def estabilidad_pct(pct_variacion, z_score):
    """100 = sin cambios atípicos; baja hacia 0 mientras más se aleje de lo normal."""
    base = 100.0
    if not pd.isna(pct_variacion):
        if pct_variacion in (float("inf"), float("-inf")):
            return 0.0
        base = min(base, 100.0 - min(100.0, K_PCT * abs(pct_variacion)))
    if not pd.isna(z_score):
        base = min(base, 100.0 - min(100.0, K_Z * abs(z_score)))
    return max(0.0, round(base, 2))


combinado = variacion_valor.merge(
    variacion_vol, on=["serie", "periodo_contable_analisis"], how="outer",
    suffixes=("_valor", "_volumen"),
)
combinado = combinado.merge(
    df[["serie", "periodo_contable_analisis", "total_registros"]].drop_duplicates(),
    on=["serie", "periodo_contable_analisis"], how="left",
)

combinado["estabilidad"] = combinado.apply(
    lambda r: min(
        estabilidad_pct(r["pct_variacion_valor"], r["z_score_valor"]),
        estabilidad_pct(r["pct_variacion_volumen"], r["z_score_volumen"]),
    ),
    axis=1,
)
combinado["atipica"] = (
    combinado["variacion_atipica_valor"].fillna(False)
    | combinado["variacion_atipica_volumen"].fillna(False)
)
combinado["cantidad_mala"] = combinado.apply(
    lambda r: int(r["total_registros"]) if r["atipica"] and not pd.isna(r["total_registros"]) else 0,
    axis=1,
)

print(f"{len(combinado)} combinaciones fuente+cuenta+libro+mes con indicador de variación combinado")
combinado[["serie", "periodo_contable_analisis", "estabilidad", "atipica", "cantidad_mala"]].head(10)

### 5.1 Armar la tabla `filas` que lee la plantilla

La plantilla espera una lista plana de filas con: `tipo_indicador`, `nombre_campo`, `periodo_contable`,
`porcentaje`, `cantidad_mala`, `total_registros`, `es_total`. Por cada indicador armamos el detalle
(una fila por combinación) y un total por periodo (para el aro y la tendencia).

In [ ]:
filas = []

# Completitud: detalle por combinación
for _, r in df.iterrows():
    filas.append({
        "tipo_indicador": "completitud",
        "nombre_campo": r["serie"],
        "periodo_contable": int(r["periodo_contable_analisis"]),
        "porcentaje": float(r["pct_completitud"]),
        "cantidad_mala": int(r["sin_valor"]),
        "total_registros": int(r["total_registros"]),
        "es_total": False,
    })

# Completitud: total por periodo (ponderado por registros)
for p in PERIODOS:
    dp = df[df["periodo_contable_analisis"] == p]
    if dp.empty:
        continue
    tot = int(dp["total_registros"].sum())
    con = int(dp["con_valor"].sum())
    filas.append({
        "tipo_indicador": "completitud",
        "nombre_campo": "TOTAL_PERIODO",
        "periodo_contable": int(p),
        "porcentaje": round(100.0 * con / tot, 2) if tot else 0.0,
        "cantidad_mala": int(dp["sin_valor"].sum()),
        "total_registros": tot,
        "es_total": True,
    })

# Variación (estabilidad combinada valor + volumen): detalle por combinación
for _, r in combinado.dropna(subset=["estabilidad"]).iterrows():
    filas.append({
        "tipo_indicador": "variacion",
        "nombre_campo": r["serie"],
        "periodo_contable": int(r["periodo_contable_analisis"]),
        "porcentaje": float(r["estabilidad"]),
        "cantidad_mala": int(r["cantidad_mala"]),
        "total_registros": int(r["total_registros"]) if not pd.isna(r["total_registros"]) else 0,
        "es_total": False,
    })

# Variación: total por periodo (promedio simple de estabilidad de esa ventana)
for p in PERIODOS:
    dp = combinado[combinado["periodo_contable_analisis"] == p].dropna(subset=["estabilidad"])
    if dp.empty:
        continue
    filas.append({
        "tipo_indicador": "variacion",
        "nombre_campo": "TOTAL_PERIODO",
        "periodo_contable": int(p),
        "porcentaje": round(float(dp["estabilidad"].mean()), 2),
        "cantidad_mala": int(dp["cantidad_mala"].sum()),
        "total_registros": int(dp["total_registros"].sum()),
        "es_total": True,
    })

# Disponibilidad de carga: 1 fila por periodo (no tiene "detalle", ya es un total)
series_totales = df["serie"].nunique()
for p in PERIODOS:
    presentes = df[df["periodo_contable_analisis"] == p]["serie"].nunique()
    huecos_p = int((huecos["periodo_contable_analisis"] == p).sum())
    filas.append({
        "tipo_indicador": "disponibilidad",
        "nombre_campo": "disponibilidad_carga",
        "periodo_contable": int(p),
        "porcentaje": round(100.0 * presentes / series_totales, 2) if series_totales else 0.0,
        "cantidad_mala": huecos_p,
        "total_registros": series_totales,
        "es_total": True,
    })

print(f"{len(filas)} filas armadas para el tablero")

In [ ]:
DATA = {
    "periodos": PERIODOS,
    "periodo_en_curso": ULTIMO_PERIODO,
    "filas": filas,
    "target": None,
    "umbrales": {
        # Completitud: casi no debería haber nulos.
        "completitud":    {"BUENO": 99.5, "ALERTA": 98, "GRAVE": 95, "CRÍTICO": 95},
        # Variación: calibrado para que 80 coincida justo con los umbrales de la
        # sección 4 (20% de cambio vs. mes anterior, o 2 desviaciones estándar).
        "variacion":      {"BUENO": 95,   "ALERTA": 80, "GRAVE": 60, "CRÍTICO": 60},
        # Disponibilidad de carga: pequeños huecos puntuales son tolerables.
        "disponibilidad": {"BUENO": 99,   "ALERTA": 95, "GRAVE": 90, "CRÍTICO": 90},
        # Usado solo para el aro "Comportamiento general" (promedio de los indicadores).
        "default":        {"BUENO": 95,   "ALERTA": 90, "GRAVE": 80, "CRÍTICO": 80},
    },
    "diccionario": {},
    "generado": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
    "fuente": "Liberty.RESERVAS.DIRECTA_RESERVA_INTERFAZ, CEDIDAS_RESERVA_INTERFAZ, "
              "CEDIDAS_RESERVA_INTERFAZ_IAXIS, CEDIDAS_TERREMOTO_RESERVA_INTERFAZ (VALOR_RESERVA_CONTABLE)",
}

### 5.2 Ensamblar el archivo final

La plantilla (`tablero_template.html`) ya trae todo el HTML/CSS/JS — lo único que hace falta es
reemplazar el texto `__DATA_JSON__` por los datos de arriba, convertidos a JSON.

In [ ]:
import json
from pathlib import Path

data_json = json.dumps(DATA, ensure_ascii=False)

plantilla = Path("tablero_template.html").read_text(encoding="utf-8")
html_doc = plantilla.replace("__DATA_JSON__", data_json)

ruta_html = Path("dashboard_completitud.html").resolve()
ruta_html.write_text(html_doc, encoding="utf-8")
print(f"Tablero guardado en: {ruta_html}")

## 6. Dónde quedó el resultado

El archivo `dashboard_completitud.html` (guardado junto a este notebook) es un tablero interactivo
autocontenido — no necesita Python ni conexión al DWH para verse: ábrelo con doble clic en cualquier
navegador. Tiene 5 pestañas:

- **Resumen ejecutivo**: aros de Completitud, Variación (Estabilidad) y Comportamiento general (DQ
  Score), tarjeta de Disponibilidad de carga, tendencia y distribución de registros afectados.
- **Detalle por combinación**: tabla fuente · cuenta · libro por indicador, de peor a mejor.
- **Tendencia e historia**: evolución de los 12 meses y variación vs. el mes anterior.
- **Análisis de alertas**: distribución por indicador y Pareto 80/20 de las combinaciones más
  afectadas.
- **Información**: qué mide cada indicador, umbrales, y las notas metodológicas del control.

Haz clic en cualquier aro del resumen para ver el detalle de qué falla ahí mismo (modal de
drill-down) — igual que en los otros tableros de calidad de datos del equipo.

Recordatorios:
- Cuentas excluidas por diseño: Incurrido, Salvamentos, Recobros, IVA AG — **+ 2 cuentas pendientes de
  confirmar con Andrey**.
- Los umbrales (`UMBRAL_Z_VARIACION`, `UMBRAL_PCT_VARIACION`, y los umbrales de estado en `DATA["umbrales"]`)
  son una primera propuesta — todavía hay que validarlos con el equipo (es la "Fase 2" mencionada en
  `Hallazgos y Estado del Proyecto.md`).
- Si cambias el diseño visual, edita `notebooks/tablero_template.html` (no este notebook) — así el
  notebook se queda solo con la lógica de datos.